In [2]:
import multiprocessing
import os
import time
from pynq.overlays.base import BaseOverlay
import socket

## to access buttons
base = BaseOverlay("base.bit")
btns = base.btns_gpio

MY_IP_ADDRESS = "192.168.8.199"
SEAN_ADDRESS = "192.168.8.225"

# LAPTOP_ADDRESS = "192.168.2.1"
MY_PYNQ = "192.168.8.207"

SERVER_PORT = 5356
CLIENT_PORT = 4376

In [3]:
%%microblaze base.PMODB

#include "gpio.h"

gpio pin_out = gpio_open(0);

void select_pin(unsigned int pin)
{
    pin_out = gpio_open(pin);
    gpio_set_direction(pin_out,GPIO_OUT);
}

int write_gpio(unsigned int pin, unsigned int val){
    gpio_write(pin_out,val);
    return 1;
    
}

In [6]:
procs = [] # a future list of all our processes

def buzz(freq,duration,pin):
    select_pin(pin)
    start=time.time()
    while start+duration>time.time():
        ret = write_gpio(pin,1)
        time.sleep(1/(freq*2))
        ret = write_gpio(pin,0)
        time.sleep(1/(freq*2))

def start_server(_server_address,_server_port):
    sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
    connect = True
    msg = ''
    # TODO:
    sock.bind((_server_address,_server_port))# 1: Bind the socket to the pynq board <CLIENT-IP> at port <LISTENING-PORT>
    sock.listen(1)# 2: Accept connections
    print("Server: waiting for client connection!")
    (clientsocket,address)= sock.accept() # 3: Receive bytes from the connection
    while connect: 
        msg = clientsocket.recv(32)
        print(msg)
        if msg == b'BUZZ!':
            buzz(2000,0.5,0)
            #print("this would start buzzer!")
        elif msg == b'Close':
            connect = False
    print("Client closed connection. server closing.....")
    clientsocket.close()

def button_check():
        button = btns.read()
        #print(f'button check value: {button}')
        return button

def start_client(_client_address,_client_port):
    sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
    connected=False

    while 1:
        time.sleep(0.5)
        state = button_check()
        match state:
            case 0x1:
                print("button 0 pressed")
                sock.connect((_client_address,_client_port))# 1: Connect the socket (sock) to the <SERVER-IP> and choosen port <LISTENING-PORT>
                time.sleep(0.5)

                connected=True   
                print(" Client connected!") 
                break
    while connected:
        state = button_check()
        match state:
            case 0x2:
                sock.send(b'BUZZ!')
                time.sleep(0.5)

                print("button 1 pressed")
            case 0x8:
                sock.send(b'Close')
                time.sleep(0.5)

                print("button 3 pressed")
                connected = False
    sock.close()
            
                    
    
    # sock.close()# 3: Close the socket, only when button is pressed

In [10]:
# Launch process1 on CPU0
p1_start = time.time()
p1 = multiprocessing.Process(target=start_server, args=(MY_PYNQ,SERVER_PORT)) # the first arg defines which CPU to run the 'target' on
# os.system("taskset -p -c {} {}".format(0, p1.pid)) # taskset is an os command to pin the process to a specific CPU
p1.start() # start the process
procs.append(p1)

p2_start = time.time()
p2 = multiprocessing.Process(target=start_client, args=(SEAN_ADDRESS,CLIENT_PORT)) # the first arg defines which CPU to run the 'target' on
# os.system("taskset -p -c {} {}".format(1, p2.pid)) # taskset is an os command to pin the process to a specific CPU
p2.start() # start the process
procs.append(p2)

p2Name = "client_process" # get process2 name
p1Name = "server_process" # get process1 name

# Here we wait for process1 to finish then wait for process2 to finish
p1.join() # wait for process1 to finish
print('Process 1 with name, {}, is finished'.format(p1Name))
p2.join() # wait for process2 to finish
print('Process 2 with name, {}, is finished'.format(p2Name))

Server: waiting for client connection!
b'BUZZ!'
b'BUZZ!'
b'Close'
Client closed connection. server closing.....
Process 1 with name, server_process, is finished
button 0 pressed
 Client connected!
button 1 pressed
button 1 pressed
button 3 pressed
Process 2 with name, client_process, is finished
